## Data preparation

Finds all audio files, splits them by recording into train / validation / test,
loads and quality-checks the audio, and saves it to `data_cache/` for the next notebooks.

Before running! change `DATA` in the first cell to the folder where you saved the sound files.

Run this notebook once. Notebooks 02 and 03 load the saved files


In [10]:
import os
import numpy as np
from scipy.io import wavfile
import pandas as pd
import matplotlib.pyplot as plt



DATA = r"F:\AI&ML\SoundFiles" # change to where you saved sounfiles.
CLASSES = {"engine1_good": 0, "engine2_broken": 1, "engine3_heavyload": 2}
CLASS_NAMES = ["good", "broken", "heavyload"]
# training on pure, validation on talk_1 and atmo_med. Testing rest
VAL_RECORDINGS = ["talking_1", "atmo_medium"]


def load_wav(path):
    sample_rate, x = wavfile.read(path)          # read wav file
    x = x.astype(np.float32) / 32768    # scale to between -1 and 1
    x = x - np.mean(x)                  # remove DC offset
    return sample_rate, x

1. Find files and split by recording

In [11]:
# Train_cut data
train_paths = []
train_labels = []

for class_folder, label in CLASSES.items():
    folder_path = os.path.join(DATA, "train_cut", class_folder)

    for filename in os.listdir(folder_path):
        train_paths.append(os.path.join(folder_path, filename))
        train_labels.append(label)

print("Train files:", len(train_paths))

Train files: 357


In [12]:
# Test_cut data
val_paths = []
val_labels = []
val_recordings = []

test_paths = []
test_labels = []
test_recordings = []

for class_folder, label in CLASSES.items():
    folder_path = os.path.join(DATA, "test_cut", class_folder)

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        recording = filename.rsplit("_", 1)[0]          # saves whole recording name.

        if recording in VAL_RECORDINGS:                  # puts validation recodings into validation set
            val_paths.append(file_path)
            val_labels.append(label)
            val_recordings.append(recording)
        else:                                            # else into test set.
            test_paths.append(file_path)
            test_labels.append(label)
            test_recordings.append(recording)


print("Val files:", len(val_paths))
print("Test files:", len(test_paths))

Val files: 442
Test files: 1579


In [13]:
# check that no recording is in both validation and test
print("Val recordings: ", set(val_recordings))
print("Test recordings:", set(test_recordings))
print("Overlap:        ", set(val_recordings) & set(test_recordings))   # should be empty: set()

Val recordings:  {'talking_1', 'atmo_medium'}
Test recordings: {'stresstest', 'talking_2', 'whitenoise_low', 'atmo_high', 'talking_3', 'atmo_low', 'talking_4'}
Overlap:         set()


In [14]:
all_recordings = np.concatenate([["pure"] * len(train_labels), val_recordings, test_recordings])
all_labels = np.concatenate([train_labels, val_labels, test_labels])

table = pd.crosstab(all_recordings, all_labels)
table.columns = CLASS_NAMES
print(table)

                good  broken  heavyload
row_0                                  
atmo_high         70      70         70
atmo_low          72      68         70
atmo_medium       70      74         70
pure             105     124        128
stresstest        41      41         41
talking_1         70      72         86
talking_2         72      71         70
talking_3         70      70         70
talking_4         70      70         70
whitenoise_low   134     129        140


The current split is like 15% training data, 20% validation and 65% Testing which is how the data was split from download

we could change it but we risk training and validating/testing on really similar audio files.
inflate our results with trained on testing data...

could mabe split it so that we 
train on 
    pure, talking1, talking2, atmo_low, atmo_high, 50%
validate on
    talking3, atmo_medium 20%
Test on
    talking4, whitenoise_low, stresstest 30%'

benefit would be more training data, but might make results unaccurate for real world application

## 2. Load audio and quality check

Checks for sample rate == 44100 and length == 3s, also if clips are slient

In [15]:
# function to load and quality check the data.

def load_split(paths, labels): 
    X = []
    y = []

    for i in range(len(paths)):
        sample_rate, x = load_wav(paths[i])               

        if sample_rate != 44100:                          # quality checks
            print("Wrong sample rate:", paths[i])
            continue                             
        if len(x) != 132300:
            print("Wrong length:", paths[i])
            continue
        if np.max(np.abs(x)) < 0.001:
            print("Silent file:", paths[i])
            continue

        X.append(x)                              # keep passed files
        y.append(labels[i])

    print("Kept", len(X), "of", len(paths), "files")
    return np.array(X), np.array(y)

In [16]:
# run cell to load training and validation data into memory.
X_train, y_train = load_split(train_paths, train_labels)
X_val, y_val = load_split(val_paths, val_labels)

print("Train:", X_train.shape, "Val:", X_val.shape)

Kept 357 of 357 files
Kept 442 of 442 files
Train: (357, 132300) Val: (442, 132300)


## 3. Save for the next notebooks
saves the files so that you dont have to read audio files each time

In [17]:
# saves the loaded audio so notebook 02 and 03 can load it in seconds
os.makedirs("data_cache", exist_ok=True)

np.save("data_cache/X_train.npy", X_train)
np.save("data_cache/y_train.npy", y_train)
np.save("data_cache/X_val.npy", X_val)
np.save("data_cache/y_val.npy", y_val)
np.save("data_cache/val_recordings.npy", np.array(val_recordings))

print("Saved train and val to data_cache/")

Saved train and val to data_cache/


## 4. Test set
Load the test set only when the models are finished and all choices are made.
Keeping it separate means it should never influence the model

In [18]:

LOAD_TEST = False # set True when ready to load test set.

if LOAD_TEST:
    X_test, y_test = load_split(test_paths, test_labels)
    test_recordings = np.array(test_recordings)

    np.save("data_cache/X_test.npy", X_test)
    np.save("data_cache/y_test.npy", y_test)
    np.save("data_cache/test_recordings.npy", test_recordings)

    print("Saved test to data_cache/")
else:
    print("Test set NOT loaded (LOAD_TEST = False)")


Test set NOT loaded (LOAD_TEST = False)
